# Task 3 – ML Model Portfolio

This notebook trains four regression models to predict taxi fare amount and records their training time, hyper‑parameters, and evaluation metrics. All modeling is performed with PySpark's MLlib.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorIndexer
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import time, os, json

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Load the prepared data from Task 2 (assume pipeline output saved as parquet)
data_path = os.path.join(os.getcwd(), 'data', 'yellow_tripdata_2025-*.parquet')
df = spark.read.parquet(data_path)
# Re‑apply the same preprocessing pipeline defined in Task 2 (duplicate code for self‑containment)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
cat_cols = ['vendor_id', 'store_and_fwd_flag']
indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec') for c in cat_cols]
num_cols = ['passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude',
                    'dropoff_longitude', 'dropoff_latitude', 'trip_duration',
                    'pickup_hour', 'pickup_day', 'pickup_month']
assembler = VectorAssembler(inputCols=num_cols + [c+'_vec' for c in cat_cols], outputCol='features_assembled')
scaler = StandardScaler(inputCol='features_assembled', outputCol='features', withMean=True, withStd=True)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
pipeline_model = pipeline.fit(df)
prepared = pipeline_model.transform(df)
# Select final dataset for modeling
model_df = prepared.select('features', 'fare_amount')
# Train‑test split (seed=42)
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)


## Helper: Evaluate and Record Metrics

In [ ]:
def evaluate_model(predictions):
    evaluator_rmse = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='rmse')
    evaluator_mae = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='mae')
    evaluator_mse = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='mse')
    evaluator_r2 = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='r2')
    return {
        'RMSE': evaluator_rmse.evaluate(predictions),
        'MAE': evaluator_mae.evaluate(predictions),
        'MSE': evaluator_mse.evaluate(predictions),
        'R2': evaluator_r2.evaluate(predictions)
    }


## Model Definitions, Hyper‑parameter Grids, and Cross‑Validation

In [ ]:
models_info = []  # will hold dicts for the summary table

# 1. Linear Regression
lr = LinearRegression(featuresCol='features', labelCol='fare_amount')
lr_grid = (ParamGridBuilder()
                    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
                    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
                    .build())
lr_cv = CrossValidator(estimator=lr,
                               estimatorParamMaps=lr_grid,
                               evaluator=RegressionEvaluator(labelCol='fare_amount', metricName='rmse'),
                               numFolds=3,
                               seed=42)
start = time.time()
lr_model = lr_cv.fit(train_df)
lr_time = time.time() - start
lr_predictions = lr_model.transform(test_df)
lr_metrics = evaluate_model(lr_predictions)
models_info.append({
    'Algorithm': 'Linear Regression',
    'Hyperparameters': str(lr_model.bestModel.extractParamMap()),
    'Training Time (s)': round(lr_time, 2),
    **lr_metrics
})

# 2. Decision Tree Regressor
dt = DecisionTreeRegressor(featuresCol='features', labelCol='fare_amount')
dt_grid = (ParamGridBuilder()
                    .addGrid(dt.maxDepth, [5, 10, 15])
                    .addGrid(dt.minInstancesPerNode, [1, 5])
                    .build())
dt_cv = CrossValidator(estimator=dt, estimatorParamMaps=dt_grid,
                               evaluator=RegressionEvaluator(labelCol='fare_amount', metricName='rmse'),
                               numFolds=3, seed=42)
start = time.time()
dt_model = dt_cv.fit(train_df)
dt_time = time.time() - start
dt_predictions = dt_model.transform(test_df)
dt_metrics = evaluate_model(dt_predictions)
models_info.append({
    'Algorithm': 'Decision Tree Regressor',
    'Hyperparameters': str(dt_model.bestModel.extractParamMap()),
    'Training Time (s)': round(dt_time, 2),
    **dt_metrics
})

# 3. Random Forest Regressor
rf = RandomForestRegressor(featuresCol='features', labelCol='fare_amount', seed=42)
rf_grid = (ParamGridBuilder()
                    .addGrid(rf.numTrees, [20, 50])
                    .addGrid(rf.maxDepth, [10, 20])
                    .build())
rf_cv = CrossValidator(estimator=rf, estimatorParamMaps=rf_grid,
                               evaluator=RegressionEvaluator(labelCol='fare_amount', metricName='rmse'),
                               numFolds=3, seed=42)
start = time.time()
rf_model = rf_cv.fit(train_df)
rf_time = time.time() - start
rf_predictions = rf_model.transform(test_df)
rf_metrics = evaluate_model(rf_predictions)
models_info.append({
    'Algorithm': 'Random Forest Regressor',
    'Hyperparameters': str(rf_model.bestModel.extractParamMap()),
    'Training Time (s)': round(rf_time, 2),
    **rf_metrics
})

# 4. Gradient‑Boosted Trees Regressor
gbt = GBTRegressor(featuresCol='features', labelCol='fare_amount', seed=42)
gbt_grid = (ParamGridBuilder()
                    .addGrid(gbt.maxIter, [20, 50])
                    .addGrid(gbt.maxDepth, [5, 10])
                    .build())
gbt_cv = CrossValidator(estimator=gbt, estimatorParamMaps=gbt_grid,
                               evaluator=RegressionEvaluator(labelCol='fare_amount', metricName='rmse'),
                               numFolds=3, seed=42)
start = time.time()
gbt_model = gbt_cv.fit(train_df)
gbt_time = time.time() - start
gbt_predictions = gbt_model.transform(test_df)
gbt_metrics = evaluate_model(gbt_predictions)
models_info.append({
    'Algorithm': 'Gradient Boosted Tree Regressor',
    'Hyperparameters': str(gbt_model.bestModel.extractParamMap()),
    'Training Time (s)': round(gbt_time, 2),
    **gbt_metrics
})

# Identify best model (lowest RMSE)
best_model = min(models_info, key=lambda x: x['RMSE'])
print('Best model by RMSE:', best_model['Algorithm'])  # <--- Screenshot this identification

## Summary Table of All Models
The table below consolidates hyper‑parameters, training time, and evaluation metrics for each algorithm.

In [ ]:
import pandas as pd
summary_df = pd.DataFrame(models_info)
summary_df = summary_df[['Algorithm', 'Hyperparameters', 'Training Time (s)', 'RMSE', 'MAE', 'MSE', 'R2']]
display(summary_df)  # <--- Screenshot this table